## Data Ingestion Patterns

### 1. Imports

In [ ]:
import json
import gzip
import requests

from pathlib import Path
from tqdm.auto import tqdm
from tokenizers import Tokenizer

### 2. Dataset Download

In [ ]:
DATASET_URL = "https://huggingface.co/datasets/allenai/c4/resolve/main"
OUT_DIR = Path("data/c4/en")

In [ ]:
def shard_url(idx: int, split: str ="train", total: int = 1024):
      """Build the HuggingFace download URL for a single C4 shard.

      Args:
          idx: Zero-based index of the shard within the split.
          split: Dataset split name, such as "train" or "validation".
          total: Number of shards the split is published as. Upstream encodes
              this in the filename, so it has to match the published layout.

      Returns:
          Fully qualified URL of the gzipped JSON Lines shard.
      """

      return f"{DATASET_URL}/en/c4-{split}.{idx:05d}-of-{total:05d}.json.gz"

In [ ]:
def fetch(url: str, out_dir: Path, chunk_size: int = 2**20):
    """Download a URL to a local path, resuming an interrupted transfer.

    The body is streamed in fixed-size chunks so a multi-hundred-megabyte
    shard never has to be held in memory. Bytes accumulate in a sibling
    ".part" file that is renamed onto the final path only after the length
    check passes, so an interrupted run cannot leave behind a truncated file
    that a later run mistakes for a complete one.

    Args:
        url: Source URL to download from.
        out_dir: Destination file path, not a directory. A ".part" sibling is
            created alongside it while the download is in flight.
        chunk_size: Bytes buffered per write, defaulting to 1 MiB.

    Raises:
        requests.HTTPError: If the server responds with an error status.
        RuntimeError: If a ranged request is answered with 200 instead of 206,
            since appending a full body onto existing bytes would corrupt the
            file, or if the final size does not match the expected length.
    """

    out_dir.parent.mkdir(parents=True, exist_ok=True)
    part = out_dir.with_name(out_dir.name + ".part")
    have = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={have}-"} if have else {}

    with requests.get(url, headers=headers, stream=True, timeout=(10, 120)) as r:
        r.raise_for_status()
        if have and r.status_code != 206:
            have = 0
            raise RuntimeError(f"Server does not support resuming downloads: {url}")
        want = int(r.headers["Content-Length"]) + have
        with open(part, "ab" if have else "wb") as f:
            for block in r.iter_content(chunk_size=chunk_size):
                f.write(block)

        got = part.stat().st_size
        if got != want:
            raise RuntimeError(f"Download incomplete: {got} != {want}")
        part.rename(out_dir)

In [ ]:
for i in range(12):
    dest = OUT_DIR / f"c4-train.{i:05d}-of-01024.json.gz"
    if not dest.exists():
        fetch(shard_url(i), dest)

### 3. Batching

In [ ]:
BATCH_SIZE = 1024 # size of one batch in records, not bytes

In [ ]:
def iter_shard(idx: int, split: str = "train", total: int = 1024):
    """Yield the records of one C4 shard already present on disk.

    The shard is decompressed and parsed lazily, one line at a time, so a
    single record is resident at any moment no matter how large the shard is.
    This is the streaming source that batching consumes.

    Args:
        idx: Zero-based index of the shard within the split.
        split: Dataset split name, such as "train" or "validation".
        total: Number of shards the split is published as.

    Yields:
        One decoded JSON object per line of the shard.

    Raises:
        RuntimeError: If the shard is not present locally. Fetching is kept a
            separate, explicit step so that reading can never silently trigger
            a multi-hundred-megabyte download.
    """

    url = shard_url(idx, split, total)
    dest = OUT_DIR / f"c4-{split}.{idx:05d}-of-{total:05d}.json.gz"
    if not dest.exists():
        raise RuntimeError(f"Shard {dest} is missing; download it first with fetch({url}, {dest})")

    with gzip.open(dest, "rt") as f:
        for line in f:
            yield json.loads(line)

In [ ]:
def iter_dataset(split: str = "train", total: int = 1024):
    """Yield the records of every downloaded shard in a split, in order.

    Shards are concatenated logically rather than physically: each one is
    opened only once its predecessor is exhausted, so the whole split streams
    with a single record resident at a time. The shard count is discovered by
    counting local files, which assumes they are numbered contiguously from
    zero.

    Args:
        split: Dataset split name, such as "train" or "validation".
        total: Number of shards the split is published as.

    Yields:
        One decoded JSON object per line of each shard.

    Raises:
        RuntimeError: Propagated from iter_shard if a shard in the range is
            missing, which happens when the local numbering has a gap.
    """

    shard_count = len(list(OUT_DIR.glob(f"c4-{split}.*-of-{total:05d}.json.gz")))
    for idx in range(shard_count):
        yield from iter_shard(idx, split, total)

In [ ]:
def iter_batches(split: str = "train", total: int = 1024, batch_size: int = BATCH_SIZE):
    """Yield fixed-size batches of records from a split.

    Records are accumulated into a list and handed over once it reaches
    batch_size, so exactly one batch is held in memory at a time. The final
    batch is yielded short rather than discarded when the split does not
    divide evenly.

    Args:
        split: Dataset split name, such as "train" or "validation".
        total: Number of shards the split is published as.
        batch_size: Number of records to yield in each batch.

    Yields:
        A list of decoded JSON objects holding batch_size records, except for
        the final batch, which may be shorter.
    """

    batch = []
    for record in iter_dataset(split, total):
        batch.append(record)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [ ]:
TOKENIZER = Tokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    """Convert a batch of records into a batch of token ID lists.

    The texts are handed to the tokenizer as a single list rather than one at
    a time. The tokenizer is a Rust extension, so a bulk call pays the
    Python-to-native boundary once per batch instead of once per record and
    parallelises across texts internally, which measures about 3x faster than
    looping over encode() for a batch of this size.

    Args:
        batch: A list of decoded JSON objects holding records.

    Returns:
        A list of lists of integers, where each inner list holds the token IDs
        for one record, in the same order as the input batch.
    """

    texts = [record.get("text", "") for record in batch]
    return [encoding.ids for encoding in TOKENIZER.encode_batch(texts)]

In [ ]:
batches = iter_batches(split="train", total=1024, batch_size=BATCH_SIZE)

for batch in tqdm(batches):
    tokenized_batch = tokenize(batch)